# 3.4.5. Minh họa đặc trưng trên các giao dịch có nhãn

**Mục tiêu:** lấy các giao dịch thật từ `card_transaction.v1.csv`, tạo lại **10 Semantic Features bằng chính logic canonical của `final_pipeline`**, ghép lại ground truth theo `raw_row_id`, sau đó chọn:

- 2 fraud có dấu hiệu hành vi khá rõ;
- 1 fraud khó nhận biết trực quan;
- 2 non-fraud tương đối bình thường;
- 1 non-fraud có một số dấu hiệu trông khả nghi.

Notebook này **không gọi model, không tính risk score và không dùng prediction**.

Để không làm ảnh hưởng tới ý nghĩa của FINAL TEST, các mẫu minh họa được lấy trong **VALIDATION: 01/2019–05/2019**.

Các tiêu chí dùng để *chọn ví dụ dễ trình bày* không phải luật phát hiện fraud và không được dùng để đánh giá mô hình.

In [1]:
from pathlib import Path
import sys
import inspect
from dataclasses import fields, is_dataclass
from collections.abc import Mapping

import numpy as np
import pandas as pd

# Tự tìm project root từ vị trí notebook/kernel hiện tại.
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "final_pipeline").is_dir()
            and (candidate / "research").is_dir()
        ):
            return candidate
    raise RuntimeError(
        "Không tìm thấy project root. "
        "Hãy mở notebook từ bên trong project fraud-risk-screening."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_ROOT = PROJECT_ROOT / "final_pipeline" / "src"
DATA_PATH = (
    PROJECT_ROOT.parent
    / "fraud-risk-screening-local-data"
    / "ibm_tabformer"
    / "card_transaction.v1.csv"
)

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
print("Dataset exists:", DATA_PATH.is_file())

PROJECT_ROOT: /Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening
DATA_PATH: /Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening-local-data/ibm_tabformer/card_transaction.v1.csv
Dataset exists: True


In [2]:
# Import đúng API replay của final_pipeline.
# iter_semantic_card_blocks sử dụng canonical parsing + history + semantic-feature logic,
# tránh viết lại công thức behavioral feature trong notebook báo cáo.

from fraud_screening.replay import (
    CANONICAL_RAW_DATASET,
    iter_semantic_card_blocks,
)

SEMANTIC_FEATURES = [
    "amount_numeric",
    "time_since_previous_transaction_min",
    "transactions_last_1h",
    "amount_minus_previous_mean",
    "is_new_merchant",
    "has_prior_card_history",
    "transaction_mode",
    "location_state",
    "hour_of_day",
    "day_of_week",
]

print("Raw rows expected:", f"{CANONICAL_RAW_DATASET.row_count:,}")
print("Semantic feature count:", len(SEMANTIC_FEATURES))
print("iter_semantic_card_blocks signature:")
print(inspect.signature(iter_semantic_card_blocks))

assert DATA_PATH.is_file(), f"Không tìm thấy dataset: {DATA_PATH}"
assert len(SEMANTIC_FEATURES) == 10

Raw rows expected: 24,386,900
Semantic feature count: 10
iter_semantic_card_blocks signature:
(path: 'str | Path', *, chunksize: 'int' = 250000, project_root: 'str | Path | None' = None) -> 'Iterator[CardReplayBlock]'


## Bước 1 — Đọc ground truth và đánh dấu vùng VALIDATION

Ở bước này chỉ đọc `Year`, `Month` và `Is Fraud?`.

- `target_binary = 1` nếu `Is Fraud? == "Yes"`;
- `target_binary = 0` nếu `Is Fraud? == "No"`;
- chỉ các dòng từ **01/2019 đến hết 05/2019** được dùng làm ứng viên minh họa.

Hai mảng NumPy được dùng thay vì giữ toàn bộ DataFrame 24 triệu dòng trong RAM.

In [3]:
N_ROWS = CANONICAL_RAW_DATASET.row_count
LABEL_CHUNK_SIZE = 1_000_000

target_binary = np.empty(N_ROWS, dtype=np.int8)
is_validation = np.zeros(N_ROWS, dtype=bool)

offset = 0

with pd.read_csv(
    DATA_PATH,
    usecols=["Year", "Month", "Is Fraud?"],
    chunksize=LABEL_CHUNK_SIZE,
) as reader:
    for chunk in reader:
        n = len(chunk)
        end = offset + n

        target_binary[offset:end] = (
            chunk["Is Fraud?"]
            .astype("string")
            .str.strip()
            .eq("Yes")
            .astype(np.int8)
            .to_numpy()
        )

        is_validation[offset:end] = (
            chunk["Year"].eq(2019)
            & chunk["Month"].between(1, 5)
        ).to_numpy()

        offset = end

assert offset == N_ROWS

print("Rows read:", f"{offset:,}")
print("VALIDATION rows:", f"{is_validation.sum():,}")
print(
    "VALIDATION fraud:",
    f"{target_binary[is_validation].sum():,}",
)

# Canonical expected values của project.
assert is_validation.sum() == 712_458
assert target_binary[is_validation].sum() == 1_052

print("GROUND-TRUTH / VALIDATION GATE: PASS")

Rows read: 24,386,900
VALIDATION rows: 712,458
VALIDATION fraud: 1,052
GROUND-TRUTH / VALIDATION GATE: PASS


## Bước 2 — Chuẩn hóa output của canonical replay

`final_pipeline` đã có `iter_semantic_card_blocks`. Cell dưới đây chỉ làm nhiệm vụ **đọc output của API** về dạng DataFrame để notebook có thể lọc và trình bày.

Nó không tính lại `recency`, `velocity`, `previous mean` hay `merchant novelty`.

In [4]:
from dataclasses import asdict, fields, is_dataclass
from collections.abc import Mapping, Sequence


CHUNK_SIZE = 250_000


def semantic_replay():
    """
    Gọi đúng canonical replay của final_pipeline.
    """
    params = inspect.signature(
        iter_semantic_card_blocks
    ).parameters

    kwargs = {}

    if "chunksize" in params:
        kwargs["chunksize"] = CHUNK_SIZE

    if "project_root" in params:
        kwargs["project_root"] = PROJECT_ROOT

    return iter_semantic_card_blocks(
        DATA_PATH,
        **kwargs,
    )


def semantic_row_to_dict(row):
    """
    Chuyển một semantic row do final_pipeline trả về
    thành dict mà không tính lại bất kỳ feature nào.
    """

    # Trường hợp row vốn đã là dict / mapping.
    if isinstance(row, Mapping):
        return dict(row)

    # Trường hợp row là dataclass.
    if is_dataclass(row):
        return asdict(row)

    # Trường hợp namedtuple.
    if hasattr(row, "_asdict"):
        return dict(row._asdict())

    # Một số object có hàm chuyển sang mapping.
    if hasattr(row, "to_mapping"):
        return dict(row.to_mapping())

    if hasattr(row, "as_mapping"):
        return dict(row.as_mapping())

    # Object Python thông thường.
    if hasattr(row, "__dict__"):
        return dict(vars(row))

    # Trường hợp semantic row chính là sequence 10 giá trị
    # theo đúng SEMANTIC_FEATURES.
    if (
        isinstance(row, Sequence)
        and not isinstance(row, (str, bytes))
        and len(row) == len(SEMANTIC_FEATURES)
    ):
        return dict(
            zip(
                SEMANTIC_FEATURES,
                row,
            )
        )

    raise TypeError(
        "Không biết cách chuyển semantic row sang dict. "
        f"Type nhận được: {type(row)}"
    )


def unpack_semantic_block(block):
    """
    Tách raw_row_id và 10 Semantic Features
    trực tiếp từ CardReplayBlock canonical.
    """

    # CardReplayBlock đã cung cấp lineage trực tiếp.
    raw_row_ids = np.asarray(
        block.raw_row_ids,
        dtype=np.int64,
    )

    semantic_rows = block.semantic_rows

    # Nếu implementation trả thẳng DataFrame.
    if isinstance(semantic_rows, pd.DataFrame):
        semantic = semantic_rows.copy()

    else:
        semantic = pd.DataFrame(
            [
                semantic_row_to_dict(row)
                for row in semantic_rows
            ]
        )

    missing = [
        feature
        for feature in SEMANTIC_FEATURES
        if feature not in semantic.columns
    ]

    if missing:
        raise RuntimeError(
            "semantic_rows thiếu các feature: "
            f"{missing}\n"
            f"Các cột hiện có: {list(semantic.columns)}"
        )

    semantic = (
        semantic[SEMANTIC_FEATURES]
        .copy()
        .reset_index(drop=True)
    )

    if len(raw_row_ids) != len(semantic):
        raise RuntimeError(
            "Số raw_row_id không khớp số semantic row: "
            f"{len(raw_row_ids)} != {len(semantic)}"
        )

    return raw_row_ids, semantic


print("Canonical replay adapter ready.")

Canonical replay adapter ready.


## Bước 3 — Kiểm tra một block trước khi quét toàn bộ dataset

Chạy cell này trước. Mục đích là xác nhận version `final_pipeline` hiện tại trả về đúng:

- 10 Semantic Features;
- `raw_row_id` để ghép lại ground truth.

Cell chỉ lấy block đầu tiên rồi dừng.

In [5]:
probe_iter = semantic_replay()
probe_block = next(probe_iter)

print("Block type:", type(probe_block))
print(
    "Dataclass fields:",
    [
        field.name
        for field in fields(probe_block)
    ],
)

print()
print("card_key:")
print(probe_block.card_key)

print()
print("raw_row_ids type:")
print(type(probe_block.raw_row_ids))

print("Số raw_row_ids:")
print(len(probe_block.raw_row_ids))

print()
print("semantic_rows type:")
print(type(probe_block.semantic_rows))

print("Số semantic_rows:")
print(len(probe_block.semantic_rows))

# Xem đúng một semantic row trước khi chuyển đổi.
if len(probe_block.semantic_rows) > 0:
    first_semantic_row = (
        probe_block.semantic_rows[0]
    )

    print()
    print("First semantic row type:")
    print(type(first_semantic_row))

    print()
    print("First semantic row:")
    print(first_semantic_row)


probe_ids, probe_semantic = (
    unpack_semantic_block(
        probe_block
    )
)

print()
print(
    "Rows in first block:",
    len(probe_semantic),
)

print(
    "First raw_row_id:",
    int(probe_ids[0]),
)

print()
print("Semantic columns:")
print(
    list(
        probe_semantic.columns
    )
)

display(
    probe_semantic.head(3)
)

assert (
    list(probe_semantic.columns)
    == SEMANTIC_FEATURES
)

assert (
    len(probe_ids)
    == len(probe_semantic)
)

print()
print(
    "CANONICAL SEMANTIC REPLAY GATE: PASS"
)

Block type: <class 'fraud_screening.replay.card_blocks.CardReplayBlock'>
Dataclass fields: ['card_key', 'raw_row_ids', 'transactions', 'semantic_rows']

card_key:
(0, 0)

raw_row_ids type:
<class 'tuple'>
Số raw_row_ids:
5011

semantic_rows type:
<class 'tuple'>
Số semantic_rows:
5011

First semantic row type:
<class 'fraud_screening.features.semantic.SemanticFeatureRow'>

First semantic row:
SemanticFeatureRow(amount_numeric=134.09, time_since_previous_transaction_min=None, transactions_last_1h=0, amount_minus_previous_mean=None, is_new_merchant=True, has_prior_card_history=False, transaction_mode='Swipe Transaction', location_state='PHYSICAL_COMPLETE', hour_of_day='6', day_of_week='6')

Rows in first block: 5011
First raw_row_id: 0

Semantic columns:
['amount_numeric', 'time_since_previous_transaction_min', 'transactions_last_1h', 'amount_minus_previous_mean', 'is_new_merchant', 'has_prior_card_history', 'transaction_mode', 'location_state', 'hour_of_day', 'day_of_week']


,amount_numeric,time_since_previous_transaction_min,transactions_last_1h,amount_minus_previous_mean,is_new_merchant,has_prior_card_history,transaction_mode,location_state,hour_of_day,day_of_week
0,134.09,NaN,0,NaN,True,False,Swipe Transaction,PHYSICAL_COMPLETE,6,6
1,38.48,21.0,1,-95.610,True,True,Swipe Transaction,PHYSICAL_COMPLETE,6,6
2,120.34,1420.0,0,34.055,False,True,Swipe Transaction,PHYSICAL_COMPLETE,6,0



CANONICAL SEMANTIC REPLAY GATE: PASS


## Bước 4 — Tạo đầy đủ tập VALIDATION ở mức Semantic Feature

Khác với notebook cũ chỉ giữ toàn bộ fraud và lấy mẫu non-fraud, bước này giữ **toàn bộ 712.458 giao dịch VALIDATION**.

Mục đích là tạo được bảng thống kê chính xác giữa:

- các giao dịch có ground truth Fraud;
- các giao dịch có ground truth Non-fraud.

Việc so sánh này dùng để trả lời câu hỏi: **những Behavioral Feature nào thực sự có biểu hiện khác nhau đáng chú ý trong nhóm fraud?**

In [6]:
validation_parts = []

card_blocks = 0
semantic_rows_seen = 0

for block in semantic_replay():
    raw_ids, semantic = unpack_semantic_block(block)

    card_blocks += 1
    semantic_rows_seen += len(semantic)

    valid_mask = is_validation[raw_ids]

    if not valid_mask.any():
        continue

    ids_v = raw_ids[valid_mask]

    frame_v = (
        semantic.loc[valid_mask]
        .copy()
        .reset_index(drop=True)
    )

    frame_v.insert(
        0,
        "raw_row_id",
        ids_v,
    )

    frame_v["target_binary"] = (
        target_binary[ids_v]
    )

    validation_parts.append(frame_v)

    if card_blocks % 500 == 0:
        print(
            "Card blocks:",
            f"{card_blocks:,}",
            "| semantic rows:",
            f"{semantic_rows_seen:,}",
        )

validation_pool = (
    pd.concat(
        validation_parts,
        ignore_index=True,
    )
    .sort_values("raw_row_id")
    .reset_index(drop=True)
)

fraud_pool = (
    validation_pool.loc[
        validation_pool["target_binary"].eq(1)
    ]
    .copy()
    .reset_index(drop=True)
)

normal_pool = (
    validation_pool.loc[
        validation_pool["target_binary"].eq(0)
    ]
    .copy()
    .reset_index(drop=True)
)

print()
print("VALIDATION rows:", f"{len(validation_pool):,}")
print("Fraud:", f"{len(fraud_pool):,}")
print("Non-fraud:", f"{len(normal_pool):,}")

assert len(validation_pool) == 712_458
assert len(fraud_pool) == 1_052
assert len(normal_pool) == 711_406

print("FULL VALIDATION SEMANTIC POOL: PASS")

Card blocks: 500 | semantic rows: 2,021,584
Card blocks: 1,500 | semantic rows: 5,915,634
Card blocks: 2,000 | semantic rows: 7,883,446
Card blocks: 2,500 | semantic rows: 9,842,479
Card blocks: 3,000 | semantic rows: 11,897,259
Card blocks: 3,500 | semantic rows: 13,997,374
Card blocks: 4,000 | semantic rows: 16,040,304
Card blocks: 4,500 | semantic rows: 18,115,490
Card blocks: 5,000 | semantic rows: 19,945,815
Card blocks: 6,000 | semantic rows: 23,844,247

VALIDATION rows: 712,458
Fraud: 1,052
Non-fraud: 711,406
FULL VALIDATION SEMANTIC POOL: PASS


## Bước 5 — Phân tích các dấu hiệu hành vi trong nhóm Fraud

Bảng dưới đây **không tạo luật fraud**. Nó chỉ mô tả và đối chiếu một số feature có thể diễn giải được giữa Fraud và Non-fraud.

Các thống kê được chọn để dễ đưa vào báo cáo:

- trung vị chênh lệch Amount so với lịch sử;
- tỷ lệ giao dịch có Amount cao hơn mức trung bình trước đó;
- tỷ lệ có giao dịch trước trong vòng 60 phút;
- tỷ lệ có ít nhất 2 giao dịch trong một giờ trước;
- tỷ lệ merchant mới;
- tỷ lệ có lịch sử thẻ trước đó.

Mốc 60 phút gắn trực tiếp với feature `transactions_last_1h`; không phải ngưỡng của model.

In [7]:
def pct(mask):
    return float(mask.mean() * 100.0)


def median_or_nan(series):
    values = pd.to_numeric(
        series,
        errors="coerce",
    )
    return float(values.median())


behavior_summary = pd.DataFrame(
    [
        {
            "Dấu hiệu / thống kê":
                "Trung vị chênh lệch Amount so với TB trước",
            "Feature":
                "amount_minus_previous_mean",
            "Fraud":
                median_or_nan(
                    fraud_pool[
                        "amount_minus_previous_mean"
                    ]
                ),
            "Non-fraud":
                median_or_nan(
                    normal_pool[
                        "amount_minus_previous_mean"
                    ]
                ),
            "Đơn vị": "Amount",
        },
        {
            "Dấu hiệu / thống kê":
                "Tỷ lệ Amount hiện tại > TB trước",
            "Feature":
                "amount_minus_previous_mean > 0",
            "Fraud":
                pct(
                    fraud_pool[
                        "amount_minus_previous_mean"
                    ].gt(0)
                ),
            "Non-fraud":
                pct(
                    normal_pool[
                        "amount_minus_previous_mean"
                    ].gt(0)
                ),
            "Đơn vị": "%",
        },
        {
            "Dấu hiệu / thống kê":
                "Tỷ lệ có GD trước trong ≤ 60 phút",
            "Feature":
                "time_since_previous_transaction_min",
            "Fraud":
                pct(
                    fraud_pool[
                        "time_since_previous_transaction_min"
                    ].le(60)
                ),
            "Non-fraud":
                pct(
                    normal_pool[
                        "time_since_previous_transaction_min"
                    ].le(60)
                ),
            "Đơn vị": "%",
        },
        {
            "Dấu hiệu / thống kê":
                "Tỷ lệ có ≥ 2 GD trong 1 giờ trước",
            "Feature":
                "transactions_last_1h",
            "Fraud":
                pct(
                    fraud_pool[
                        "transactions_last_1h"
                    ].ge(2)
                ),
            "Non-fraud":
                pct(
                    normal_pool[
                        "transactions_last_1h"
                    ].ge(2)
                ),
            "Đơn vị": "%",
        },
        {
            "Dấu hiệu / thống kê":
                "Tỷ lệ merchant mới",
            "Feature":
                "is_new_merchant",
            "Fraud":
                pct(
                    fraud_pool[
                        "is_new_merchant"
                    ].astype(bool)
                ),
            "Non-fraud":
                pct(
                    normal_pool[
                        "is_new_merchant"
                    ].astype(bool)
                ),
            "Đơn vị": "%",
        },
        {
            "Dấu hiệu / thống kê":
                "Tỷ lệ có lịch sử thẻ trước đó",
            "Feature":
                "has_prior_card_history",
            "Fraud":
                pct(
                    fraud_pool[
                        "has_prior_card_history"
                    ].astype(bool)
                ),
            "Non-fraud":
                pct(
                    normal_pool[
                        "has_prior_card_history"
                    ].astype(bool)
                ),
            "Đơn vị": "%",
        },
    ]
)

behavior_summary["Fraud"] = (
    behavior_summary["Fraud"].round(3)
)
behavior_summary["Non-fraud"] = (
    behavior_summary["Non-fraud"].round(3)
)

display(behavior_summary)

,Dấu hiệu / thống kê,Feature,Fraud,Non-fraud,Đơn vị
0,Trung vị chênh lệch Amount so với TB trước,amount_minus_previous_mean,-3.698,-8.951,Amount
1,Tỷ lệ Amount hiện tại > TB trước,amount_minus_previous_mean > 0,47.529,40.133,%
2,Tỷ lệ có GD trước trong ≤ 60 phút,time_since_previous_transaction_min,18.821,19.411,%
3,Tỷ lệ có ≥ 2 GD trong 1 giờ trước,transactions_last_1h,2.757,6.144,%
4,Tỷ lệ merchant mới,is_new_merchant,54.183,2.495,%
5,Tỷ lệ có lịch sử thẻ trước đó,has_prior_card_history,100.000,99.995,%


### Kiểm tra thêm transaction mode và location state

Hai bảng dưới đây giúp quyết định xem `transaction_mode` hoặc `location_state` có nên được nhắc tới trong phần diễn giải hay không.

**Không nên** chỉ vì một category xuất hiện nhiều ở Fraud mà gọi nó là “dấu hiệu fraud”; cần nhìn đồng thời tỷ lệ trong Non-fraud.

In [8]:
def category_compare(column):
    fraud_dist = (
        fraud_pool[column]
        .astype("string")
        .value_counts(normalize=True)
        .mul(100)
        .rename("Fraud (%)")
    )

    normal_dist = (
        normal_pool[column]
        .astype("string")
        .value_counts(normalize=True)
        .mul(100)
        .rename("Non-fraud (%)")
    )

    result = (
        pd.concat(
            [fraud_dist, normal_dist],
            axis=1,
        )
        .fillna(0)
        .reset_index()
        .rename(columns={"index": column})
    )

    result["Fraud (%)"] = (
        result["Fraud (%)"].round(3)
    )
    result["Non-fraud (%)"] = (
        result["Non-fraud (%)"].round(3)
    )

    return result


print("TRANSACTION MODE")
display(
    category_compare(
        "transaction_mode"
    )
)

print("LOCATION STATE")
display(
    category_compare(
        "location_state"
    )
)

TRANSACTION MODE


,transaction_mode,Fraud (%),Non-fraud (%)
0,Chip Transaction,91.16,70.573
1,Swipe Transaction,8.84,17.015
2,Online Transaction,0.0,12.412


LOCATION STATE


,location_state,Fraud (%),Non-fraud (%)
0,PHYSICAL_ZIP_UNAVAILABLE,100.0,0.624
1,PHYSICAL_COMPLETE,0.0,86.877
2,NON_PHYSICAL_OR_ONLINE,0.0,12.499


## Bước 6 — Chọn các giao dịch thực tế để minh họa

Không cần đưa quá nhiều mẫu. Với mục tiêu báo cáo, bốn trường hợp thường đủ:

- **A — Fraud rõ:** nhiều dấu hiệu cùng xuất hiện;
- **B — Fraud rõ nhưng pattern khác A:** tránh chọn hai giao dịch gần như giống nhau;
- **C — Fraud khó nhìn:** cho thấy không có luật đơn giản;
- **D — Non-fraud trông khả nghi:** làm phản ví dụ, tránh suy luận “có dấu hiệu = fraud”.

Cell dưới đây tạo một số candidate để chọn thủ công. Việc chọn mẫu chỉ phục vụ **minh họa**, không tham gia huấn luyện hay đánh giá model.

In [9]:
# Candidate 1: fraud có Amount lệch lớn + merchant mới + giao dịch gần.
fraud_amount_new_recent = (
    fraud_pool.loc[
        fraud_pool["is_new_merchant"].astype(bool)
        & fraud_pool[
            "time_since_previous_transaction_min"
        ].le(60)
    ]
    .sort_values(
        [
            "amount_minus_previous_mean",
            "transactions_last_1h",
        ],
        ascending=[False, False],
        na_position="last",
    )
    .head(10)
)

# Candidate 2: fraud có nhịp giao dịch dày nhưng không nhất thiết Amount lệch lớn nhất.
fraud_velocity = (
    fraud_pool.loc[
        fraud_pool[
            "transactions_last_1h"
        ].ge(2)
    ]
    .assign(
        abs_delta=lambda x:
            x["amount_minus_previous_mean"].abs()
    )
    .sort_values(
        [
            "transactions_last_1h",
            "abs_delta",
        ],
        ascending=[False, True],
        na_position="last",
    )
    .head(10)
    .drop(columns=["abs_delta"])
)

# Candidate 3: fraud khó nhận biết bằng các Behavioral Feature đang xét.
fraud_hard = (
    fraud_pool.loc[
        (~fraud_pool["is_new_merchant"].astype(bool))
        & fraud_pool[
            "transactions_last_1h"
        ].eq(0)
        & fraud_pool[
            "time_since_previous_transaction_min"
        ].gt(120)
    ]
    .assign(
        abs_delta=lambda x:
            x["amount_minus_previous_mean"].abs()
    )
    .sort_values(
        [
            "abs_delta",
            "time_since_previous_transaction_min",
        ],
        ascending=[True, False],
        na_position="last",
    )
    .head(10)
    .drop(columns=["abs_delta"])
)

# Candidate 4: non-fraud nhưng có nhiều dấu hiệu trông khả nghi.
normal_suspicious = (
    normal_pool.loc[
        normal_pool[
            "time_since_previous_transaction_min"
        ].le(60)
        & normal_pool[
            "transactions_last_1h"
        ].ge(2)
        & normal_pool[
            "amount_minus_previous_mean"
        ].gt(0)
    ]
    .sort_values(
        [
            "amount_minus_previous_mean",
            "transactions_last_1h",
        ],
        ascending=[False, False],
        na_position="last",
    )
    .head(10)
)

CANDIDATE_COLUMNS = [
    "raw_row_id",
    "amount_numeric",
    "amount_minus_previous_mean",
    "time_since_previous_transaction_min",
    "transactions_last_1h",
    "is_new_merchant",
    "transaction_mode",
    "location_state",
]

print("A — FRAUD: Amount lệch + merchant mới + giao dịch gần")
display(fraud_amount_new_recent[CANDIDATE_COLUMNS])

print("B — FRAUD: nhịp giao dịch dày")
display(fraud_velocity[CANDIDATE_COLUMNS])

print("C — FRAUD: khó nhận biết trực quan")
display(fraud_hard[CANDIDATE_COLUMNS])

print("D — NON-FRAUD: trông khả nghi")
display(normal_suspicious[CANDIDATE_COLUMNS])

A — FRAUD: Amount lệch + merchant mới + giao dịch gần


,raw_row_id,amount_numeric,amount_minus_previous_mean,time_since_previous_transaction_min,transactions_last_1h,is_new_merchant,transaction_mode,location_state
938,21356193,540.16,496.022279,12.0,1,True,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
169,3882742,522.40,467.649659,18.0,1,True,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
987,21984571,376.38,342.108549,59.0,1,True,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
774,18070370,330.35,298.970148,29.0,1,True,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
242,5756905,288.14,212.864810,46.0,1,True,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
1006,22140228,240.84,169.857657,10.0,3,True,Swipe Transaction,PHYSICAL_ZIP_UNAVAILABLE
642,16045068,208.35,169.810803,23.0,1,True,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
650,16203064,179.30,162.380862,54.0,1,True,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
418,9798669,182.50,137.511382,17.0,1,True,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
1007,22140229,198.76,127.739469,29.0,3,True,Swipe Transaction,PHYSICAL_ZIP_UNAVAILABLE


B — FRAUD: nhịp giao dịch dày


,raw_row_id,amount_numeric,amount_minus_previous_mean,time_since_previous_transaction_min,transactions_last_1h,is_new_merchant,transaction_mode,location_state
419,9798673,10.57,-34.482593,5.0,5,False,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
100,2236585,-86.00,-119.933852,4.0,4,False,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
626,14656444,49.68,-9.071346,22.0,3,True,Swipe Transaction,PHYSICAL_ZIP_UNAVAILABLE
254,6074095,86.00,9.969614,3.0,3,False,Swipe Transaction,PHYSICAL_ZIP_UNAVAILABLE
375,9266896,38.30,20.505987,5.0,3,True,Swipe Transaction,PHYSICAL_ZIP_UNAVAILABLE
440,10532390,66.70,26.914746,5.0,3,True,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
414,9798664,98.54,53.676948,11.0,3,True,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
918,20738434,-62.00,-74.476304,3.0,3,False,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
1007,22140229,198.76,127.739469,29.0,3,True,Swipe Transaction,PHYSICAL_ZIP_UNAVAILABLE
1006,22140228,240.84,169.857657,10.0,3,True,Swipe Transaction,PHYSICAL_ZIP_UNAVAILABLE


C — FRAUD: khó nhận biết trực quan


,raw_row_id,amount_numeric,amount_minus_previous_mean,time_since_previous_transaction_min,transactions_last_1h,is_new_merchant,transaction_mode,location_state
824,18723122,55.82,0.467856,148.0,0,False,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
785,18295307,34.22,0.771054,2626.0,0,False,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
199,4869242,65.39,-0.934108,2691.0,0,False,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
769,17526112,54.89,-1.118179,250.0,0,False,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
1036,23961006,35.15,-1.472516,184.0,0,False,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
521,12209589,73.37,-1.642028,164.0,0,False,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
653,16203073,15.08,-1.883270,4269.0,0,False,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
328,8688045,69.60,-1.949245,448.0,0,False,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
895,19470935,21.37,2.140822,9655.0,0,False,Chip Transaction,PHYSICAL_ZIP_UNAVAILABLE
473,11024547,20.87,-2.506379,284.0,0,False,Swipe Transaction,PHYSICAL_ZIP_UNAVAILABLE


D — NON-FRAUD: trông khả nghi


,raw_row_id,amount_numeric,amount_minus_previous_mean,time_since_previous_transaction_min,transactions_last_1h,is_new_merchant,transaction_mode,location_state
605050,20807852,2169.75,2116.002165,9.0,3,True,Online Transaction,NON_PHYSICAL_OR_ONLINE
279796,9580222,1725.34,1637.793985,3.0,4,False,Chip Transaction,PHYSICAL_COMPLETE
152627,5303561,1530.71,1447.088637,7.0,2,False,Chip Transaction,PHYSICAL_COMPLETE
381224,13228780,1227.92,1162.606106,7.0,2,False,Chip Transaction,PHYSICAL_COMPLETE
494359,17075312,1129.46,1079.420512,24.0,2,False,Chip Transaction,PHYSICAL_COMPLETE
404585,14036686,1155.20,1075.458833,44.0,3,False,Online Transaction,NON_PHYSICAL_OR_ONLINE
232584,8023851,1076.22,1020.240463,1.0,2,False,Swipe Transaction,PHYSICAL_COMPLETE
70118,2457744,993.68,926.173033,26.0,2,False,Chip Transaction,PHYSICAL_COMPLETE
282680,9674087,856.41,815.522200,36.0,2,False,Online Transaction,NON_PHYSICAL_OR_ONLINE
623752,21413045,833.74,781.486101,16.0,3,False,Chip Transaction,PHYSICAL_COMPLETE


## Bước 7 — Có thể tận dụng trực tiếp output notebook cũ

Notebook cũ đã tìm được bốn mẫu rất hữu ích:

```text
A — raw_row_id 22140228 — Fraud
Amount = 240.84
Amount - TB trước = +169.86
Phút từ GD trước = 10
GD trong 1h trước = 3
Merchant mới = Có

B — raw_row_id 22140229 — Fraud
Amount = 198.76
Amount - TB trước = +127.74
Phút từ GD trước = 29
GD trong 1h trước = 3
Merchant mới = Có

C — raw_row_id 18723122 — Fraud
Amount = 55.82
Amount - TB trước = +0.47
Phút từ GD trước = 148
GD trong 1h trước = 0
Merchant mới = Không

F — raw_row_id 2313900 — Non-fraud
Amount = 406.00
Amount - TB trước = +347.65
Phút từ GD trước = 2
GD trong 1h trước = 2
Merchant mới = Không
```

A và B có pattern khá giống nhau. Sau khi chạy candidate ở bước 6, nên cân nhắc thay B bằng một fraud có pattern khác để bảng minh họa đa dạng hơn.

## Bước 8 — Bảng minh họa đề xuất cho báo cáo

Sau khi chọn bốn `raw_row_id`, tạo bảng với cấu trúc:

```markdown
| Mẫu | Ground truth | Amount | Chênh lệch với TB trước | Phút từ GD trước | GD trong 1h trước | Merchant mới | Mode / Location | Nhận xét |
|---|---|---:|---:|---:|---:|---|---|---|
```

Bảng này chỉ trình bày dữ liệu thật và các feature đã xây dựng. **Không đưa risk score hoặc prediction của Random Forest vào mục 3.4.5.**

## Định hướng cho demo thực nghiệm

Sau phần phân tích và minh họa, có thể rút ra định hướng cho demo:

```text
Demo không cần sao chép một giao dịch fraud có sẵn
→ xây một tình huống giả lập dễ hiểu
→ có lịch sử thẻ bình thường
→ giao dịch hiện tại kết hợp 2–3 dấu hiệu hành vi
→ đưa qua pipeline thật
→ quan sát risk score và screening result
```

Ví dụ định hướng:

- lịch sử trước có Amount tương đối ổn định;
- giao dịch hiện tại có Amount tăng rõ so với lịch sử;
- merchant hiện tại là merchant mới;
- có thể thêm thay đổi về location hoặc transaction mode;
- không cần cố tình làm mọi feature đều “bất thường”.

Điểm quan trọng khi viết báo cáo:

> Demo là **tình huống được xây dựng để kiểm tra pipeline**, không có ground truth fraud và không được mô tả như một giao dịch gian lận đã được xác nhận.